[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/01_collection/A4_apn_enrichment.ipynb)

# A4: APN Enrichment

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Understand APNs** - why Assessor Parcel Numbers are the gold standard for property identification
2. **Match addresses to APNs** using county assessor data
3. **Handle edge cases** - projects with only names, unusual addresses, or multiple parcels
4. **Validate APN formats** for your county
5. **Enrich projects** with parcel attributes (owner, zoning, lot size)

## Why APNs Matter

**Problem:** Addresses are inconsistent
- "123 Main St" vs "123 Main Street" vs "123 MAIN ST APT 1"
- Projects may span multiple addresses
- New construction has no address yet

**Solution:** Assessor Parcel Numbers (APNs)
- Unique identifier for every parcel in the county
- Never changes (unless parcels split/merge)
- Links to ownership, tax, and zoning records
- Required for official HCD reporting

## APN Format (Alameda County)

```
Format: XXX-XXXX-XXX-XX
Example: 056-2034-001-00

  056  = Map book number
 2034  = Map page number  
  001  = Parcel number
   00  = Sub-parcel (condos, etc.)
```

---

## 1. Environment Setup

In [ ]:
# ============================================================================
# COLAB ENVIRONMENT SETUP (Run this first in Colab!)
# ============================================================================

import os
import sys
from pathlib import Path

print('SETTING UP ENVIRONMENT')
print('='*70)

# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

if IN_COLAB:
    # Clone repository
    repo_path = Path('/content/berkeley-housing-analysis')
    
    if not repo_path.exists():
        print('\nCloning repository...')
        !git clone https://github.com/blockXblock/berkeley-housing-analysis.git
        print('Repository cloned')
    else:
        print('\nRepository already exists')
        !cd /content/berkeley-housing-analysis && git pull
    
    os.chdir(repo_path)
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))
    
    ROOT = repo_path
    print(f'\nWorking directory: {os.getcwd()}')
else:
    print(f'\nWorking directory: {os.getcwd()}')

print('\n' + '='*70)
print('SETUP COMPLETE!')
print('='*70)

In [ ]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path

# Find project root
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve paths
DATA_DIR = ROOT / CONFIG['paths']['data_dir']
LOOKUP_TABLE = ROOT / CONFIG['paths']['lookup_table']
HOUSING_CSV = ROOT / CONFIG['paths']['housing_projects']

print(f"Project root: {ROOT}")
print(f"Lookup table: {LOOKUP_TABLE}")
print(f"Housing data: {HOUSING_CSV}")

In [ ]:
# Import project modules
from modules.geocoder import normalize_address_for_lookup, geocode_from_lookup
from modules.address_normalizer import normalize_address

## 2. Assess Current APN Coverage

In [ ]:
# Load housing projects
df = pd.read_csv(HOUSING_CSV)
print(f"Loaded {len(df)} projects")

# APN coverage statistics
has_apn = df['apn'].notna() & (df['apn'] != '') & (df['apn'] != 'nan')
has_address = df['address_display'].notna()

print(f"\n=== Project Identification Coverage ===")
print(f"Has APN:           {has_apn.sum():>4} ({100*has_apn.sum()/len(df):.1f}%)")
print(f"Has Address:       {has_address.sum():>4} ({100*has_address.sum()/len(df):.1f}%)")
print(f"Has Both:          {(has_apn & has_address).sum():>4}")
print(f"Missing APN:       {(~has_apn).sum():>4}")
print(f"Missing Address:   {(~has_address).sum():>4}")

In [ ]:
# Show projects missing APNs
missing_apn = df[~has_apn][['address_display', 'apn', 'description', 'net_units', 'status']]

if len(missing_apn) > 0:
    print(f"Projects missing APN ({len(missing_apn)} total):")
    display(missing_apn)
else:
    print("All projects have APNs!")

## 3. Validate APN Format

Alameda County APNs follow the format: `XXX-XXXX-XXX-XX`

Common issues:
- Missing leading zeros (56-2034-1-0 should be 056-2034-001-00)
- Missing hyphens
- Extra characters

In [ ]:
def validate_alameda_apn(apn):
    """
    Validate Alameda County APN format.
    Returns (is_valid, normalized_apn, error_message)
    """
    if pd.isna(apn) or str(apn).strip() == '' or str(apn) == 'nan':
        return False, None, 'Empty APN'
    
    apn_str = str(apn).strip()
    
    # Standard format: XXX-XXXX-XXX-XX
    standard_pattern = r'^(\d{3})-(\d{4})-(\d{3})-(\d{2})$'
    match = re.match(standard_pattern, apn_str)
    
    if match:
        return True, apn_str, None
    
    # Try to normalize common variations
    # Remove all non-digit characters first
    digits_only = re.sub(r'[^\d]', '', apn_str)
    
    if len(digits_only) == 12:
        # Reformat to standard
        normalized = f"{digits_only[:3]}-{digits_only[3:7]}-{digits_only[7:10]}-{digits_only[10:12]}"
        return True, normalized, 'Reformatted'
    
    if len(digits_only) < 12:
        return False, apn_str, f'Too few digits ({len(digits_only)})'
    
    if len(digits_only) > 12:
        return False, apn_str, f'Too many digits ({len(digits_only)})'
    
    return False, apn_str, 'Unknown format'


# Validate all APNs
validation_results = df[has_apn]['apn'].apply(validate_alameda_apn)
df.loc[has_apn, 'apn_valid'] = [r[0] for r in validation_results]
df.loc[has_apn, 'apn_normalized'] = [r[1] for r in validation_results]
df.loc[has_apn, 'apn_error'] = [r[2] for r in validation_results]

# Summary
valid_count = df['apn_valid'].sum()
invalid_count = has_apn.sum() - valid_count

print(f"=== APN Validation Results ===")
print(f"Valid APNs:   {int(valid_count)}")
print(f"Invalid APNs: {int(invalid_count)}")

if invalid_count > 0:
    print(f"\nInvalid APNs:")
    invalid = df[df['apn_valid'] == False][['address_display', 'apn', 'apn_error']]
    display(invalid)

## 4. Find Missing APNs from Address Lookup

Use the Alameda County address lookup table (563K addresses) to find APNs for projects that have addresses but are missing APNs.

In [ ]:
# Load lookup table
if LOOKUP_TABLE.exists():
    df_lookup = pd.read_csv(LOOKUP_TABLE)
    print(f"Loaded {len(df_lookup):,} addresses from lookup table")
    print(f"Columns: {df_lookup.columns.tolist()}")
    
    # Sample
    print(f"\nSample entries:")
    display(df_lookup.head(3))
else:
    print(f"Lookup table not found at: {LOOKUP_TABLE}")
    df_lookup = None

In [ ]:
def lookup_apn_by_address(address, df_lookup):
    """
    Look up APN from address using the Alameda County table.
    Returns (apn, confidence, match_address) or (None, None, None)
    """
    if df_lookup is None or pd.isna(address):
        return None, None, None
    
    # Normalize the input address
    normalized = normalize_address_for_lookup(str(address))
    
    if normalized is None:
        return None, None, None
    
    # Try exact match
    matches = df_lookup[df_lookup['normalized_address'] == normalized]
    
    if len(matches) == 1:
        row = matches.iloc[0]
        return row['APN'], 'exact', row['original_address']
    
    if len(matches) > 1:
        # Multiple matches - return first but flag as ambiguous
        row = matches.iloc[0]
        return row['APN'], 'ambiguous', row['original_address']
    
    # No exact match - try fuzzy match on street number + name
    parts = str(address).strip().split()
    if len(parts) >= 2:
        street_num = parts[0]
        street_name = parts[1].upper()
        
        # Filter by street number
        candidates = df_lookup[df_lookup['street_number'].astype(str) == street_num]
        
        if len(candidates) > 0:
            # Find best match by street name
            for idx, row in candidates.iterrows():
                if street_name in str(row['street_name']).upper():
                    return row['APN'], 'partial', row['original_address']
    
    return None, None, None


# Test on a known address
if df_lookup is not None:
    test_addr = df[has_apn].iloc[0]['address_display']
    test_apn, confidence, match = lookup_apn_by_address(test_addr, df_lookup)
    print(f"Test lookup: '{test_addr}'")
    print(f"  Found APN: {test_apn}")
    print(f"  Confidence: {confidence}")
    print(f"  Matched: {match}")

In [ ]:
# Try to fill missing APNs
if df_lookup is not None:
    missing_mask = ~has_apn & has_address
    
    if missing_mask.sum() > 0:
        print(f"Attempting to find APNs for {missing_mask.sum()} projects...\n")
        
        for idx in df[missing_mask].index:
            address = df.loc[idx, 'address_display']
            apn, confidence, match = lookup_apn_by_address(address, df_lookup)
            
            if apn:
                print(f"FOUND: '{address}'")
                print(f"  APN: {apn} ({confidence} match)")
                df.loc[idx, 'apn'] = apn
                df.loc[idx, 'apn_source'] = f'lookup_{confidence}'
            else:
                print(f"NOT FOUND: '{address}'")
                df.loc[idx, 'apn_source'] = 'manual_needed'
    else:
        print("No projects need APN lookup!")

## 5. Handle Project-Name-Only Records

Some projects are known only by name (e.g., "The Ashby BART Project") before an address is assigned.

Strategy:
1. Extract location hints from project name/description
2. Cross-reference with known developments
3. Mark for manual research

In [ ]:
def extract_location_hints(text):
    """
    Extract potential location hints from project name or description.
    """
    if pd.isna(text):
        return []
    
    text = str(text).upper()
    hints = []
    
    # Known Berkeley landmarks/areas
    landmarks = [
        'ASHBY', 'DOWNTOWN', 'NORTH BERKELEY', 'SOUTH BERKELEY',
        'UNIVERSITY', 'SHATTUCK', 'TELEGRAPH', 'SAN PABLO',
        'BART', 'CAMPUS', 'MARINA', 'AQUATIC PARK',
        'ELMWOOD', 'ROCKRIDGE', 'CLAREMONT', 'THOUSAND OAKS'
    ]
    
    for landmark in landmarks:
        if landmark in text:
            hints.append(landmark)
    
    # Look for street names
    street_pattern = r'\b(\d+)\s+(\w+)\s+(ST|AVE|BLVD|WAY|RD|DR|CT|PL)\b'
    matches = re.findall(street_pattern, text)
    for match in matches:
        hints.append(f"{match[0]} {match[1]} {match[2]}")
    
    return hints


# Find projects that might need location hints
needs_location = df[~has_apn | ~has_address].copy()

if len(needs_location) > 0:
    print(f"Projects needing location research ({len(needs_location)}):")
    
    for idx, row in needs_location.iterrows():
        hints_from_desc = extract_location_hints(row.get('description', ''))
        hints_from_addr = extract_location_hints(row.get('address_display', ''))
        all_hints = list(set(hints_from_desc + hints_from_addr))
        
        print(f"\n  Project: {row.get('address_display', row.get('description', 'Unknown')[:50])}")
        print(f"  Units: {row.get('net_units', 'N/A')}")
        if all_hints:
            print(f"  Location hints: {', '.join(all_hints)}")
        else:
            print(f"  Location hints: None found - manual research needed")
else:
    print("All projects have addresses!")

## 6. Geocode Fallback: Find APN from Coordinates

If we have coordinates but no APN, we can find the containing parcel.

**Note:** This requires a parcel boundary shapefile or API. For demonstration, we'll show the approach.

In [ ]:
def find_nearest_apn_from_coords(lat, lon, df_lookup, max_distance_m=50):
    """
    Find the nearest APN to given coordinates.
    Uses simple distance calculation (Haversine approximation).
    
    For production, use a proper spatial database or parcel boundary shapefile.
    """
    if df_lookup is None or pd.isna(lat) or pd.isna(lon):
        return None, None
    
    # Filter to nearby addresses (rough bounding box)
    lat_tol = 0.001  # ~111 meters
    lon_tol = 0.001  # ~85 meters at Berkeley's latitude
    
    nearby = df_lookup[
        (df_lookup['latitude'].between(lat - lat_tol, lat + lat_tol)) &
        (df_lookup['longitude'].between(lon - lon_tol, lon + lon_tol))
    ].copy()
    
    if len(nearby) == 0:
        return None, None
    
    # Calculate approximate distance
    nearby['dist'] = np.sqrt(
        ((nearby['latitude'] - lat) * 111000)**2 +
        ((nearby['longitude'] - lon) * 85000)**2
    )
    
    # Get closest
    closest = nearby.loc[nearby['dist'].idxmin()]
    
    if closest['dist'] <= max_distance_m:
        return closest['APN'], closest['dist']
    
    return None, None


# Example: Find APNs for projects with coords but no APN
if df_lookup is not None:
    has_coords = df['latitude'].notna() & df['longitude'].notna()
    needs_apn_has_coords = ~has_apn & has_coords
    
    if needs_apn_has_coords.sum() > 0:
        print(f"Trying coordinate-based APN lookup for {needs_apn_has_coords.sum()} projects...\n")
        
        for idx in df[needs_apn_has_coords].index:
            lat = df.loc[idx, 'latitude']
            lon = df.loc[idx, 'longitude']
            address = df.loc[idx, 'address_display']
            
            apn, dist = find_nearest_apn_from_coords(lat, lon, df_lookup)
            
            if apn:
                print(f"FOUND: '{address}' @ ({lat:.4f}, {lon:.4f})")
                print(f"  Nearest APN: {apn} ({dist:.1f}m away)")
            else:
                print(f"NOT FOUND: '{address}' @ ({lat:.4f}, {lon:.4f})")
    else:
        print("No projects need coordinate-based APN lookup.")

## 7. Manual APN Research

For projects that can't be matched automatically, use these resources:

### Alameda County Resources
1. **Assessor Parcel Viewer**: https://gis.acgov.org/Html5Viewer/Index.html?viewer=parcels
2. **Property Search**: https://www.acgov.org/assessor/propertysearch.htm

### Berkeley Resources
1. **Berkeley GIS**: https://www.cityofberkeley.info/gis/
2. **Planning Records**: https://www.cityofberkeley.info/Planning_and_Development/

### Workflow for Manual Lookup
1. Open Alameda County Parcel Viewer
2. Navigate to the project location
3. Click on the parcel to get APN
4. Enter below

In [ ]:
# Manual APN entries
# Add APNs you've researched manually here

MANUAL_APNS = {
    # 'address or project name': 'XXX-XXXX-XXX-XX',
    # Example:
    # '0 LE ROY Ave': '057-1234-001-00',
}

# Apply manual entries
for address_key, apn_value in MANUAL_APNS.items():
    matches = df[df['address_display'].str.contains(address_key, case=False, na=False)]
    if len(matches) > 0:
        for idx in matches.index:
            df.loc[idx, 'apn'] = apn_value
            df.loc[idx, 'apn_source'] = 'manual'
            print(f"Applied manual APN {apn_value} to {address_key}")

if not MANUAL_APNS:
    print("No manual APNs defined. Add entries to MANUAL_APNS dict as needed.")

## 8. Final Coverage Report

In [ ]:
# Recalculate coverage
has_apn_final = df['apn'].notna() & (df['apn'] != '') & (df['apn'] != 'nan')

print("="*60)
print("FINAL APN COVERAGE REPORT")
print("="*60)
print(f"")
print(f"Total projects:    {len(df)}")
print(f"Has APN:           {has_apn_final.sum()} ({100*has_apn_final.sum()/len(df):.1f}%)")
print(f"Missing APN:       {(~has_apn_final).sum()}")
print(f"")

# Units coverage
units_with_apn = df[has_apn_final]['net_units'].sum()
units_total = df['net_units'].sum()
print(f"Units with APN:    {units_with_apn:,.0f} of {units_total:,.0f} ({100*units_with_apn/units_total:.1f}%)")
print(f"")

# Source breakdown
if 'apn_source' in df.columns:
    print("APN Sources:")
    print(df['apn_source'].value_counts().to_string())

In [ ]:
# Show any remaining projects without APNs
still_missing = df[~has_apn_final][['address_display', 'description', 'net_units', 'status', 'latitude', 'longitude']]

if len(still_missing) > 0:
    print(f"\nProjects still missing APNs ({len(still_missing)}):")
    display(still_missing)
    
    print("\nAction needed: Use manual research workflow (Section 7)")
else:
    print("\nAll projects now have APNs!")

## 9. Export Updated Data

In [ ]:
# Save updated projects
output_path = DATA_DIR / 'housing_projects_with_apn.csv'

# Select columns to export (exclude temporary validation columns)
export_cols = [c for c in df.columns if c not in ['apn_valid', 'apn_error']]
df[export_cols].to_csv(output_path, index=False)

print(f"Saved {len(df)} projects to {output_path}")
print(f"APN coverage: {has_apn_final.sum()}/{len(df)} ({100*has_apn_final.sum()/len(df):.1f}%)")

---

## Summary

This notebook:
1. Assessed current APN coverage
2. Validated APN formats for Alameda County
3. Looked up missing APNs from address table
4. Identified projects needing manual research
5. Provided coordinate-based fallback lookup
6. Generated final coverage report

**Next:** Run `A5_buildingeye_import.ipynb` to add timeline data, or proceed to `B1_lifecycle_tracking.ipynb`.

---

## Adapting for Other Counties

| County | APN Format | Lookup Resource |
|--------|------------|----------------|
| Alameda | XXX-XXXX-XXX-XX | acgov.org/assessor |
| Contra Costa | XXX-XXX-XXX | ccmap.us |
| San Francisco | XXXX-XXX | sfassessor.org |
| Santa Clara | XXX-XX-XXX | sccassessor.org |

Modify `validate_alameda_apn()` for your county's format.